In [10]:
try:
    spark.stop()
except Exception as e:
    pass

# ⚙️ 1. Setup Spark local (avec support Delta)


In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("RecipeSearchPoC")
    # Support Delta Lake
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # Broadcast join automatique pour df_index (table d'index souvent < 200 Mo)
    .config("spark.sql.autoBroadcastJoinThreshold", "200m")
    # Gestion de la mémoire occupée par Spark
    .config("spark.executor.memory", "2g")  # Mémoire allouée à chaque nœud de calcul
    .config("spark.driver.memory", "2g")    # Mémoire allouée au chef d'orchestre
    .getOrCreate()
)

print("✅ Spark ready")


✅ Spark ready


# 📥 2. Chargement des données


In [12]:
DATA_PATH = "../data/recipes_parquets"

# Delta gère nativement le Data Skipping et le Z-Order définis dans le pipeline
df_main      = spark.read.format("delta").load(f"{DATA_PATH}/recipes_main")
df_index     = spark.read.format("delta").load(f"{DATA_PATH}/ingredients_index")
df_nutrition = spark.read.format("delta").load(f"{DATA_PATH}/recipes_nutrition_detail")

print("✅ Data loaded (lazy — aucun scan déclenché)")


✅ Data loaded (lazy — aucun scan déclenché)


# 🔗 3. Vue master


In [13]:
# On définit le join une seule fois — Spark ne l'exécute qu'à la demande
df_master = df_main.join(df_nutrition, on="recipe_id", how="left")

print(f"Colonnes master ({len(df_master.columns)}) : {df_master.columns}")


Colonnes master (22) : ['recipe_id', 'title', 'description', 'instructions_text', 'ingredients_raw', 'ingredients_validated', 'n_ingredients_validated', 'n_steps', 'cook_minutes', 'cook_time_category', 'image_url', 'image_urls', 'has_image', 'source_url', 'energy_kcal', 'nutri_score', 'tags', 'fat_g', 'protein_g', 'salt_g', 'saturates_g', 'sugars_g']


# 🔎 4. Recherche par nom


In [14]:
def search_by_name(query: str, limit: int = 20) -> list[dict]:
    rows = (
        df_master
        .filter(col("title").ilike(f"%{query}%"))
        .select("recipe_id", "title", "cook_minutes", "nutri_score", "image_url")
        .limit(limit)
        .collect()
    )
    return [r.asDict() for r in rows]

# Test
search_by_name("pasta")


,recipe_id,title,cook_minutes,nutri_score,image_url
0,c7372c374e,Homemade Fettuccini Pasta Dough,NaN,E,https://s-media-cache-ak0.pinimg.com/736x/92/4...
1,e53039454b,Egyptian Koshary Pasta,NaN,E,https://thumbs.dreamstime.com/z/egyptian-food-...
2,7b22c46b8e,Greek Orzo Pasta Salad,20.0,E,http://www.hummusapien.com/wp-content/uploads/...
3,309ca5c971,Cucumber Pasta,NaN,E,"http://cdn.skim.gs/image/upload/c_fill,dpr_1.0..."
4,27085e38d7,Crabstick pasta salad,NaN,E,https://img-global.cpcdn.com/001_recipes/58397...
5,0e90bd4380,Hearty Whole Wheat Pasta With Brussels Sprouts...,NaN,E,http://graphics8.nytimes.com/images/2013/03/04...
6,4a08b98086,Garlic Chickpea Pasta,NaN,E,http://tastykitchen.com/recipes/wp-content/upl...
7,2fef790bbc,Greek Pasta Recipe,NaN,E,https://www.hy-vee.com/webres/Image/Recipes/Gr...
8,ba6e749d43,Creamy Southwestern BBQ Pasta Salad,NaN,E,https://resources.salewhale.ca/brand_lists_ima...
9,1b97c3a5a8,Creamy Three Cheese Pasta,NaN,E,http://www.recipe-diaries.com/wp-content/uploa...


# 🥕 5. Recherche par ingrédient (le cœur)


In [15]:
def search_by_ingredient(ingredient):
    """
    df_index est typiquement < 200 Mo → AQE le broadcast automatiquement.
    Le join devient un map-side join, pas de shuffle sur df_master.
    """
    recipe_ids = (
        df_index
        .filter(col("ingredient").ilike(f"%{ingredient}%"))
        .select("recipe_id")
        .distinct()
    )
    return (
        df_master
        .join(recipe_ids, on="recipe_id")
        .select("recipe_id", "title", "cook_minutes", "nutri_score", "image_url", "ingredients_raw")
        .limit(20)
    )

# Test
search_by_ingredient("tomato").toPandas()


,recipe_id,title,cook_minutes,nutri_score,image_url,ingredients_raw
0,2f08b721c7,Grilled Mediterranean Kebabs on Pita,72.0,E,"http://img.sndimg.com/food/image/upload/w_512,...",[1 12 lbs ground lamb (may use up to 50% groun...
1,f62b577671,Harira (lamb and Bean Soup),NaN,E,http://l7.alamy.com/zooms/2f088f602bf7455fbe94...,"[12 lb diced lamb or 12 lb dice stewing beef, ..."
2,ef9a3f2778,Ham-Tomato-Pineapple-Cheese Pie,NaN,E,"http://img.sndimg.com/food/image/upload/w_512,...","[1 pie shell, baked, 1 -2 cup ham (deli meat i..."
3,6016fcc710,Greek Chicken Salad with Whole Wheat Pita and ...,NaN,E,http://www.fnstatic.co.uk/images/content/recip...,"[3 tablespoon extra-virgin olive oil, plus oil..."
4,d68edb812a,Grilled Tomato Pepper Salsa,NaN,E,http://www.floatingkitchen.net/wp-content/uplo...,"[2 medium-size ripe tomatoes, 1 red bell peppe..."
5,1fce2862d6,Grilled Skirt Steak Gyros,NaN,E,https://s-media-cache-ak0.pinimg.com/236x/4f/a...,"[1/3 cup extra-virgin olive oil, 3 cloves garl..."
6,1aceba59c7,Grilled Ratatouille,NaN,E,https://www.cleaneatingmag.com/.image/t_share/...,"[1 large onion, quartered, 4 -5 garlic cloves,..."
7,4e7de99979,"Healthy Food Christmas Tree, for Any Time of Y...",NaN,E,https://i.pinimg.com/236x/81/29/89/812989d17e3...,"[1 red bell pepper, 1 green bell pepper, 1 yel..."
8,e4f8654725,Easy Chicken & Cheese Enchiladas,NaN,E,http://kiwiandbean.com/wp-content/uploads/2014...,[1 (10 3/4 ounce) can Campbell's Cream of Chic...
9,33957f7a84,Fragrant Vegetables,NaN,E,https://ae01.alicdn.com/kf/HTB1i_0SMpXXXXaNXFX...,"[2 tablespoons olive oil, 3 large onions, slic..."


# 🧠 6. Recherche avancée (clé du PoC)


In [16]:
def search_advanced(name=None, ingredient=None, max_time=None, nutri_score=None):
    """
    Les filtres sont poussés avant le join (predicate pushdown).
    Spark n'ouvre que les fichiers Delta concernés.
    """
    df = df_master

    if name:
        df = df.filter(col("title").ilike(f"%{name}%"))

    if max_time:
        df = df.filter(col("cook_minutes") <= max_time)

    if nutri_score:
        df = df.filter(col("nutri_score") == nutri_score)

    # Le join sur df_index vient EN DERNIER pour profiter des filtres précédents
    if ingredient:
        recipe_ids = (
            df_index
            .filter(col("ingredient").ilike(f"%{ingredient}%"))
            .select("recipe_id")
            .distinct()
        )
        df = df.join(recipe_ids, on="recipe_id")

    return df.select(
        "recipe_id",
        "title",
        "cook_minutes",
        "cook_time_category",
        "nutri_score",
        "energy_kcal",
        "image_url",
        "ingredients_raw",
    ).limit(20)

# Test : recettes rapides avec nutri_score A
search_advanced(max_time=30, nutri_score="A").toPandas()


,recipe_id,title,cook_minutes,cook_time_category,nutri_score,energy_kcal,image_url,ingredients_raw
0,22e5882cfb,Better Frizzled Cabbage,20,rapide,A,46.700001,http://www.annsentitledlife.com/wp-content/upl...,"[1 (16 ounce) bag coleslaw mix, 2 slices bacon..."
1,0ad9232f41,Marinade for Steak,15,rapide,A,46.500000,"http://img.sndimg.com/food/image/upload/w_512,...","[1 tablespoon olive oil, 2 tablespoons soy sau..."
2,fae646ea12,Strawberry Blender Blast,5,rapide,A,53.701080,http://2.bp.blogspot.com/-IM-LgvwuZzU/Un83kHMA...,"[1 12 cups fresh halved strawberries, 12 cup s..."
3,b15859ee59,No Sugar Fruit Cookies,25,rapide,A,57.400002,http://www.fashionfoodmusic.com/wp-content/upl...,"[1 cup chopped dates, 12 cup finely chopped pe..."
4,e08c4f00dc,Butterbeer,2,rapide,A,45.483196,"http://img.sndimg.com/food/image/upload/w_512,...","[1 cup brown sugar, 4 tablespoons butter, 12 t..."
5,d42d98bb81,Steamed Broccoli,3,rapide,A,34.000000,"http://img.sndimg.com/food/image/upload/w_512,...","[1 12 lbs broccoli, salt and pepper, 3 tablesp..."
6,bcbae7110a,Spicy Zucchini,10,rapide,A,57.099998,http://neighborfoodblog.com/wp-content/uploads...,"[1 lb zucchini, 3 tablespoons olive oil, 2 ear..."
7,93f9cf59df,Cajun Tuna Spread,5,rapide,A,62.400002,https://www.weightwatchers.com/images/1033/dyn...,"[13 cup cream cheese, softened, 3 tablespoons ..."
8,747f6d0de1,Mango Lassi Smoothie,10,rapide,A,53.941128,https://silkroutechef.files.wordpress.com/2013...,"[1 ripe mango, 12 cup vanilla yogurt, 14 cup 1..."
9,7660ce921b,Quick Homemade Tomato Soup,17,rapide,A,79.099998,"http://img.sndimg.com/food/image/upload/w_512,...","[2 tablespoons butter, 2 tablespoons olive oil..."


# 🍽️ 7. Affichage recette complète


In [17]:
def show_recipe(recipe_id):
    """
    .limit(1).collect() = Spark s'arrête dès le 1er match,
    pas de scan complet de la table.
    """
    recipe = (
        df_master
        .filter(col("recipe_id") == recipe_id)
        .limit(1)
        .collect()
    )

    if not recipe:
        print("❌ Recette introuvable")
        return

    r = recipe[0].asDict()

    print("🍽️ ", r.get("title", "—"))
    print("⏱️  Temps      :", r.get("cook_minutes"), "min  [", r.get("cook_time_category"), "]")
    print("🔥 Calories   :", r.get("energy_kcal"), "kcal")
    print("🥗 Nutri-score:", r.get("nutri_score"))
    print("🖼️  Image      :", r.get("image_url"))
    print("\n📝 Ingrédients:")
    print(r.get("ingredients_raw", "—"))
    print("\n👨‍🍳 Instructions:")
    print(r.get("instructions_text", "—"))

# Test : 1er recipe_id disponible
first_id = df_master.select("recipe_id").first()["recipe_id"]
show_recipe(first_id)


🍽️  Pumpkin Creme Caramel
⏱️  Temps      : 65 min  [ long ]
🔥 Calories   : 247.10000610351562 kcal
🥗 Nutri-score: C
🖼️  Image      : http://recipegreat.com/images/pumpkin-creme-caramel-02.jpg

📝 Ingrédients:
['23 cup plus 1/2 cup sugar, divided', '1/2 tsp. salt, divided', '4 large eggs', '1 1/4 cups pumpkin puree or pure canned pumpkin', '2 tsp. vanilla extract', '1/2 tsp. plus 18 tsp. pumpkin pie spice', '1 12-oz. can low-fat evaporated milk', '1/2 cup whole milk']

👨‍🍳 Instructions:
Preheat oven to 350F. | Lightly coat 8 6-oz. | ramekins with cooking spray, and set in 2-inch-deep baking pan. | Combine 2/3 cup sugar, 1/4 tsp. | salt, and 1 Tbs. | water in small saucepan, and bring to a boil over medium heat. | Cook 8 to 10 minutes without stirring, or until edges start to color. | Swirl pan gently to distribute browned sugar; continue to cook 1 to 2 minutes more, or until sugar is deep golden brown. | Pour 1 Tbs. | caramelized sugar into each ramekin. | Set aside. | Whisk together egg

# 🛑 8. Arrêt Spark


In [18]:
spark.stop()
print("✅ Spark stopped")


✅ Spark stopped
